<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Face_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mediapipe deepface opencv-python

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import cv2
from deepface import DeepFace
import mediapipe as mp

target_img = cv2.imread("Trump_2025.jpg")
mp_face = mp.solutions.face_detection
face_detector = mp_face.FaceDetection(model_selection=0, min_detection_confidence=0.5)

In [32]:
def get_face_crop(image, detection):
    h, w, _ = image.shape
    bbox = detection.location_data.relative_bounding_box

    x1 = int(bbox.xmin * w)
    y1 = int(bbox.ymin * h)
    x2 = int((bbox.xmin + bbox.width) * w)
    y2 = int((bbox.ymin + bbox.height) * h)

    # ensure valid bounds
    x1 = max(x1, 0)
    y1 = max(y1, 0)
    x2 = min(x2, w)
    y2 = min(y2, h)

    return image[y1:y2, x1:x2]

In [33]:
def is_match(face1, face2, threshold=0.75):
    try:
        result = DeepFace.verify(face1, face2, model_name='Facenet', enforce_detection=False)
        return result['distance'] < threshold
    except:
        return False

In [14]:
# !ls /content/drive/MyDrive

In [34]:
video_path = "/content/drive/MyDrive/videos/Trump_Air_Force_One.mp4"
video_to_use = cv2.VideoCapture(video_path)

fps = video_to_use.get(cv2.CAP_PROP_FPS)

In [35]:
import cv2

video = cv2.VideoCapture("/content/drive/MyDrive/videos/Trump_Air_Force_One.mp4")
fps = video.get(cv2.CAP_PROP_FPS)

frame_num = 0
matches = []

while True:
    ret, frame = video.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_detector.process(rgb)

    if results.detections:
        for det in results.detections:
            face_crop = get_face_crop(frame, det)

            if face_crop.size > 0 and is_match(target_img, face_crop):
                timestamp = frame_num / fps
                matches.append(timestamp)
                print(f"Match found at {timestamp:.2f} seconds!")

    frame_num += 1

video.release()

print("Done scanning video.")
print("Matches:", matches)

Done scanning video.
Matches: []
